# Lab 5.2 – Kundensegmente mit Clustering

**Beispiel (fertige Lösung).** Zellen der Reihe nach mit **Umschalt+Enter** ausführen.

Eingabe: `kunden_rfm.csv` aus dem Beispiel von Lab 2.3 (400 Kunden, Stichtag 30.09.2026). Es gibt keine Zielgröße: k-Means sucht Gruppen ähnlicher Kunden selbst. Ergebnis: `kunden_segmente.csv` und `segmente.html` in `ORDNER`.

In [1]:
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

EINGABE = "project/examples/m02-data-preparation/l03-features/kunden_rfm.csv"
ORDNER = "project/examples/m05-ml-basics/l02-clustering/"     # hier landen kunden_segmente.csv und segmente.html

kunden = pd.read_csv(EINGABE)
MERKMALE = ["tage_seit_kauf", "haeufigkeit", "umsatz"]
print(len(kunden), "Kunden")
kunden[MERKMALE].describe().round(1)

400 Kunden


,tage_seit_kauf,haeufigkeit,umsatz
count,400.0,400.0,400.0
mean,59.0,82.9,24499.4
std,105.9,93.1,38563.4
min,0.0,2.0,656.4
25%,2.0,28.0,5299.4
50%,12.0,54.5,12226.5
75%,42.0,108.0,27727.1
max,491.0,703.0,311637.3


## Schritt 1 (fertig): Skalieren

Umsatz reicht bis über 300.000 €, Häufigkeit bis etwa 700 Buchungen. Ohne Skalieren würde k-Means fast nur nach Euro gruppieren. `StandardScaler` bringt jede Spalte auf Mittelwert 0 und Standardabweichung 1.

In [2]:
skaliert = StandardScaler().fit_transform(kunden[MERKMALE])   # je Spalte: Mittelwert 0, Standardabweichung 1
pd.DataFrame(skaliert, columns=MERKMALE).describe().round(1)

,tage_seit_kauf,haeufigkeit,umsatz
count,400.0,400.0,400.0
mean,-0.0,0.0,-0.0
std,1.0,1.0,1.0
min,-0.6,-0.9,-0.6
25%,-0.5,-0.6,-0.5
50%,-0.4,-0.3,-0.3
75%,-0.2,0.3,0.1
max,4.1,6.7,7.5


## Schritt 2 (fertig): k-Means mit 4 Segmenten

`n_clusters=4` ist eine fachliche Wahl; `random_state=42` und `n_init=10` machen das Ergebnis bei jedem Lauf gleich. Danach werden die Segmente nach Ø Umsatz nummeriert: Segment 1 hat den höchsten Ø Umsatz.

In [3]:
ANZAHL_SEGMENTE = 4
kmeans = KMeans(n_clusters=ANZAHL_SEGMENTE, random_state=42, n_init=10)
kunden["segment"] = kmeans.fit_predict(skaliert)          # Nummern 0 bis 3, Reihenfolge zufällig

# Segmente nach Ø Umsatz neu nummerieren: 1 = höchster Ø Umsatz
reihenfolge = kunden.groupby("segment")["umsatz"].mean().sort_values(ascending=False).index
kunden["segment"] = kunden["segment"].map({alt: neu for neu, alt in enumerate(reihenfolge, start=1)})
kunden["segment"].value_counts().sort_index()

segment
1     14
2     90
3     55
4    241
Name: count, dtype: int64

## Schritt 3 (fertig): Kennzahlen und Streudiagramm je Segment

In [4]:
segment_tabelle = kunden.groupby("segment").agg(
    kunden=("kunde_id", "count"),
    tage_seit_kauf=("tage_seit_kauf", "mean"),
    haeufigkeit=("haeufigkeit", "mean"),
    umsatz=("umsatz", "mean"),
    umsatz_summe=("umsatz", "sum"),
).round(0)
segment_tabelle["anteil_umsatz_prozent"] = (
    segment_tabelle["umsatz_summe"] / segment_tabelle["umsatz_summe"].sum() * 100
).round(1)
segment_tabelle

,kunden,tage_seit_kauf,haeufigkeit,umsatz,umsatz_summe,anteil_umsatz_prozent
segment,,,,,,
1,14,2.0,443.0,191977.0,2687683.0,27.4
2,90,4.0,157.0,45684.0,4111553.0,42.0
3,55,302.0,54.0,13040.0,717216.0,7.3
4,241,27.0,41.0,9474.0,2283298.0,23.3


In [5]:
fig = px.scatter(
    kunden, x="haeufigkeit", y="umsatz", color=kunden["segment"].astype(str),
    hover_data=["kunde_id", "tage_seit_kauf"],
    title="14 Top-Kunden bringen gut ein Viertel des Umsatzes",
    labels={"haeufigkeit": "Buchungen bis 30.09.2026", "umsatz": "Umsatz in €", "color": "Segment"},
    category_orders={"color": ["1", "2", "3", "4"]},
)
fig

## Schritt 4: Segmente benennen (Aufgabe 1)

Die Tabelle oben zeigt je Segment Anzahl, Ø Tage seit dem letzten Kauf, Ø Buchungen und Ø Umsatz. Die Zelle darunter mit dem vorgegebenen Prompt aus der Übung (Aufgabe 3) füllen, dann die Prüfzelle ausführen.

In [6]:
# >>> AUFGABE 1: Spalte segment_name mit einem Namen je Segment ergänzen – Prompt siehe Übung, Aufgabe 3
namen = {
    1: "Top-Kunden",           # wenige Kunden, sehr viele Buchungen, höchster Umsatz
    2: "Stammkunden",          # kaufen laufend, mittlerer bis hoher Umsatz
    3: "Abgesprungen",         # letzter Kauf im Schnitt vor rund 300 Tagen
    4: "Gelegenheitskunden",   # aktiv, aber wenige Buchungen und kleiner Umsatz
}
kunden["segment_name"] = kunden["segment"].map(namen)
# <<< ENDE AUFGABE 1
kunden.groupby(["segment", "segment_name"], dropna=False).size()

segment  segment_name      
1        Top-Kunden             14
2        Stammkunden            90
3        Abgesprungen           55
4        Gelegenheitskunden    241
dtype: int64

In [7]:
# Prüfzelle – nicht ändern
def pruefe_namen(t):
    if "segment_name" not in t.columns or t["segment_name"].isna().any():
        return "nicht OK – Spalte segment_name fehlt oder ist leer: jedem Segment 1–4 einen Namen geben."
    if (t.groupby("segment")["segment_name"].nunique() != 1).any():
        return "nicht OK – ein Segment hat mehrere Namen: je Segmentnummer genau ein Name."
    if t["segment_name"].nunique() != ANZAHL_SEGMENTE:
        return f"nicht OK – {t['segment_name'].nunique()} verschiedene Namen statt {ANZAHL_SEGMENTE}: jeden Namen nur einmal vergeben."
    return "OK – " + ", ".join(f"{s} = {n}" for s, n in t.groupby("segment")["segment_name"].first().items())


print(pruefe_namen(kunden))

OK – 1 = Top-Kunden, 2 = Stammkunden, 3 = Abgesprungen, 4 = Gelegenheitskunden


## Ergebnis speichern

`kunden_segmente.csv` enthält je Kunde Segment und Name; die Segmenttabelle bekommt die Namen dazu.

In [8]:
kunden.to_csv(ORDNER + "kunden_segmente.csv", index=False)
fig.write_html(ORDNER + "segmente.html", include_plotlyjs="cdn")
print("Gespeichert:", ORDNER + "kunden_segmente.csv", "und", ORDNER + "segmente.html")
segment_tabelle["segment_name"] = kunden.groupby("segment")["segment_name"].first()
segment_tabelle

Gespeichert: project/examples/m05-ml-basics/l02-clustering/kunden_segmente.csv und project/examples/m05-ml-basics/l02-clustering/segmente.html


,kunden,tage_seit_kauf,haeufigkeit,umsatz,umsatz_summe,anteil_umsatz_prozent,segment_name
segment,,,,,,,
1,14,2.0,443.0,191977.0,2687683.0,27.4,Top-Kunden
2,90,4.0,157.0,45684.0,4111553.0,42.0,Stammkunden
3,55,302.0,54.0,13040.0,717216.0,7.3,Abgesprungen
4,241,27.0,41.0,9474.0,2283298.0,23.3,Gelegenheitskunden


## Erweiterung: Elbow-Diagramm

Für 1 bis 8 Segmente misst `inertia_`, wie weit die Kunden im Schnitt von ihrem Segment-Mittelpunkt entfernt liegen (kleiner = enger). Wo die Kurve abknickt („Ellbogen“), bringt ein weiteres Segment wenig. Danach `ANZAHL_SEGMENTE` in Schritt 2 auf 3 oder 5 setzen und ab Schritt 2 neu ausführen.

In [9]:
werte = []
for k in range(1, 9):
    modell = KMeans(n_clusters=k, random_state=42, n_init=10).fit(skaliert)
    werte.append({"segmente": k, "abstand_innen": round(modell.inertia_, 1)})
elbow = pd.DataFrame(werte)
px.line(elbow, x="segmente", y="abstand_innen", markers=True,
        title="Elbow: nach 3–4 Segmenten flacht die Kurve ab",
        labels={"segmente": "Anzahl Segmente", "abstand_innen": "Abstand innerhalb der Segmente (inertia_)"})